# 4. LLM-режим команды Graph-EVT-agent

**Цель:** запустить детерминированный `GraphEVTPipeline` и попросить LLM проверить результат относительно синтетической разметки. Для этого учебного набора `is_extreme` задаёт известную границу события и используется только для выбора чистого baseline и последующей оценки; сама метка не передаётся численному детектору.

Для живого запуска задайте `MISTRAL_API_KEY` в окружении или введите его в скрытом prompt. Ключ не записывается в ноутбук или артефакты. Использование истинной метки для определения baseline здесь oracle-assisted и не подходит для неразмеченных реальных данных.

In [1]:
from pathlib import Path
import os
from getpass import getpass
import numpy as np
import pandas as pd
from graph_evt_agent import EVTConfig, GraphConfig, GraphEVTPipeline, InputConfig
from graph_evt_agent.agents import EVTAgentTeam, MistralAPIError, MistralClient

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/generated/additive_fractal_series.csv").exists())
DATA = ROOT / "data/generated/additive_fractal_series.csv"
SEED = 42
df = pd.read_csv(DATA)
x = df["value"].to_numpy()
event_mask = df["is_extreme"].to_numpy(dtype=bool)
assert np.isfinite(x).all() and x.ndim == 1 and event_mask.shape == x.shape
assert event_mask.any(), "Generated dataset must contain a labeled event"
event_start = int(np.flatnonzero(event_mask)[0])
event_end = int(np.flatnonzero(event_mask)[-1])

In [2]:
api_key = os.getenv("MISTRAL_API_KEY") or getpass("Mistral API key (input hidden): ").strip()
MISTRAL_READY = False
mistral_client = None

if api_key:
    try:
        mistral_client = MistralClient(
            api_key=api_key,
            model="ministral-3b-2512",
            max_retries=0,
        )
        check_reply = mistral_client.complete(
            "Reply exactly with MISTRAL_OK.",
            "Connection check. Reply exactly with MISTRAL_OK.",
        )
        MISTRAL_READY = bool(check_reply.strip())
        print("Mistral API check passed." if MISTRAL_READY else "Mistral returned an empty response.")
    except MistralAPIError as error:
        error_text = str(error).lower()
        if "429" in error_text or "rate_limit" in error_text or "rate limit" in error_text:
            print("Mistral rate limit (HTTP 429). Check your account usage and rate limits, wait, then rerun this cell.")
        else:
            print(f"Mistral API check failed: {error}")
    except Exception as error:
        print(f"Mistral API check failed ({type(error).__name__}).")
else:
    print("Mistral API check skipped: no key was provided.")

Mistral API check passed.


In [3]:
# In this synthetic dataset, use the known event onset to keep the EVT baseline clean.
# The label is used for evaluation and baseline selection, not as detector input.
BASELINE_SIZE = event_start
pipeline = GraphEVTPipeline(
    EVTConfig(baseline_size=BASELINE_SIZE, tail_quantile=0.95, alarm_probability=0.999, persistence=3),
    graph=GraphConfig(random_state=SEED),
    input_config=InputConfig(missing="error"),
    verbose=True,
)
numeric_result = pipeline.run_detailed(x)
detection = numeric_result.detection
alarm_index = detection.time_index
detected_in_event = alarm_index is not None and bool(event_mask[alarm_index])
detection_delay = alarm_index - event_start if detected_in_event else None
plan = numeric_result.plan

pd.DataFrame([{
    "labeled_event_start": event_start,
    "labeled_event_end": event_end,
    "baseline_size": BASELINE_SIZE,
    "detected": detection.detected,
    "alarm_index": alarm_index,
    "detected_in_labeled_event": detected_in_event,
    "detection_delay_samples": detection_delay,
    "alarm_threshold": detection.alarm_threshold,
    "planner_route": None if plan is None else plan.route,
    "completed_actions": None if plan is None else " -> ".join(plan.completed_actions),
    "stop_reason": None if plan is None else plan.stop_reason,
}])

Graph-EVT pipeline:   0%|          | 0/3 [00:00<?, ?it/s]

,labeled_event_start,labeled_event_end,baseline_size,detected,alarm_index,detected_in_labeled_event,detection_delay_samples,alarm_threshold,planner_route,completed_actions,stop_reason
0,1000,1149,1000,True,1124,True,124,2.06665,1-d,inspect_input -> check_stationarity -> detect_evt,single_channel_cannot_localize


In [4]:
if MISTRAL_READY:
    team = EVTAgentTeam(pipeline, mistral_client)
    try:
        agent_report = team.run(
            x,
            task=(
                "Проверь результат EVT-детекции относительно синтетической разметки: "
                f"истинный интервал события [{event_start}, {event_end}], "
                f"detected={detection.detected}, alarm_index={alarm_index}, "
                f"delay_samples={detection_delay}. Укажи явно, попало ли срабатывание "
                "в размеченное событие; не называй отсутствие срабатывания доказательством "
                "отсутствия аномалии."
            ),
        )
        print(agent_report.final_report)
    except MistralAPIError as error:
        error_text = str(error).lower()
        if "429" in error_text or "rate_limit" in error_text or "rate limit" in error_text:
            print("Mistral rate limit during review. Check account usage/rate limits, wait, then rerun this cell.")
        else:
            print(f"LLM review failed: {error}")
    except Exception as error:
        print(f"LLM review failed ({type(error).__name__}).")
else:
    print("LLM review skipped; the numeric pipeline result is still available.")

Graph-EVT pipeline:   0%|          | 0/3 [00:00<?, ?it/s]

### **Краткий итог анализа данных**

#### **1. Общие параметры данных**
- **Тип анализа**: **Унивариантный (1-D)** с **1 каналом** (`channel_0`).
- **Объём данных**: **1500 точек** (без учета размерности каждого значения).
- **Пропусков**: **0%** (данные полные).

#### **2. Статистика события (EVT)**
- **Событие обнаружено**: **Да** (`detected: true`).
- **Время срабатывания**: **Точка 1124** из 1500.
- **Порог срабатывания**: **2.07** (индикатор превысил).
- **Диапазон значений**: **0.00036 – 11.37** (пик в 11.37 при пороге 2.07 → **5.55× превышение**).

#### **3. Статистическая редкость**
- **Предположительная вероятность случайности**: **0.0000124** (высокая редкость).
- **Ограничения**:
  - **Отсутствие временных меток** → невозможно точно определить реальное время события.
  - **Нерегулярность интервалов** (не указано `sampling_interval`) → потенциальные ошибки в интерпретации.
  - **Моновариантность** → невозможно локализовать событие по пространству.

#### **4. Причинная интерпр

`EVTAgentTeam` сначала вызывает тот же детерминированный pipeline. В актуальном API `run_detailed` также возвращает plan/act/observe trace; он отображён рядом с численным результатом и входит в ограниченный JSON evidence для ревьюеров. Агентам не дано права переписывать вычисления. Для 1-D входа графовый и локализационный этапы пропускаются. Текст LLM остаётся интерпретацией и должен проверяться по `numeric_result`.